In [0]:
from pyspark.sql.functions import lit, when, to_timestamp, col

In [0]:
adls_storage_options = {
    "fs.azure.account.key.nkipermeteodata001.dfs.core.windows.net": dbutils.secrets.get(
        scope="meteoschweiz", key="adls-account-key"
    )
}

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ";") \
    .options(**adls_storage_options) \
    .option("inferSchema", "true") \
    .load("abfss://data@nkipermeteodata001.dfs.core.windows.net/raw")

In [0]:
df = df.withColumn("reference_timestamp", to_timestamp(df["reference_timestamp"], "dd.MM.yyyy HH:mm"))

In [0]:
df.select("reference_timestamp").show()

In [0]:
df.printSchema()

In [0]:
value_columns = [c for c in df.columns if c not in ('station_abbr', 'reference_timestamp')]
df_long = df.unpivot(ids=['station_abbr', 'reference_timestamp'], values=value_columns, variableColumnName='parameter', valueColumnName='value')

df_long = df.unpivot(ids=['station_abbr','reference_timestamp'],values=value_columns,variableColumnName='parameter',valueColumnName='value')

In [0]:
df_long.head()

In [0]:
len(value_columns*df.count())

In [0]:
df_long.write.format("parquet") \
    .mode("overwrite") \
    .options(**adls_storage_options) \
    .save("abfss://data@nkipermeteodata001.dfs.core.windows.net/processed")

In [0]:
dim_stations = (spark.read
.format("sqlserver")
.option("host", "sqls-nkipermeteo-dev.database.windows.net") \
.option("database", "db-nkipermeteo") \
.option("dbtable", "[dim_stations]") \
.option("user", dbutils.secrets.get(scope="meteoschweiz", key="sql-username")) \
.option("password", dbutils.secrets.get(scope="meteoschweiz", key="sql-password")) \
.load()
 )

dim_parameters = (spark.read
.format("sqlserver")
.option("host", "sqls-nkipermeteo-dev.database.windows.net") \
.option("database", "db-nkipermeteo") \
.option("dbtable", "[dim_parameters]") \
.option("user", dbutils.secrets.get(scope="meteoschweiz", key="sql-username")) \
.option("password", dbutils.secrets.get(scope="meteoschweiz", key="sql-password")) \
.load()
 )

dim_date = (spark.read
.format("sqlserver")
.option("host", "sqls-nkipermeteo-dev.database.windows.net") \
.option("database", "db-nkipermeteo") \
.option("dbtable", "[dim_date]") \
.option("user", dbutils.secrets.get(scope="meteoschweiz", key="sql-username")) \
.option("password", dbutils.secrets.get(scope="meteoschweiz", key="sql-password")) \
.load()
 )

In [0]:
df_long_dim = df_long.join(
    dim_stations.select("station_abbr", "station_id"),
    on="station_abbr",
    how="left"
).join(
    dim_parameters.select(dim_parameters.parameter_shortname.alias("parameter"), "parameter_id"),
    on="parameter",
    how="left"
).join(
    dim_date.select(dim_date.full_date.cast("date").alias("day"),"date_id"),
    on=(col("day")==df_long.reference_timestamp.cast("date")),
    how="left"
).drop("day")

In [0]:
df_long_dim.count()

In [0]:
df_long_dim.head()

In [0]:
dim_parameters.toPandas().loc[dim_parameters.toPandas()['parameter_shortname'] == 'tre200d0', 'parameter_id'].values

In [0]:
max_ts_df = (spark.read
.format("sqlserver")
.option("host", "sqls-nkipermeteo-dev.database.windows.net") \
.option("database", "db-nkipermeteo") \
.option("dbtable", "(SELECT MAX(reference_timestamp) AS max_ts FROM [lf-ogd-smn_d_recent]) AS sub") \
.option("user", dbutils.secrets.get(scope="meteoschweiz", key="sql-username")) \
.option("password", dbutils.secrets.get(scope="meteoschweiz", key="sql-password")) \
.load()
 )

max_ts = max_ts_df.collect()[0][0]
print(max_ts)

In [0]:
df_long_filtered = df_long_dim.where(df_long_dim.reference_timestamp > max_ts)

In [0]:
df_long_filtered.show()

In [0]:
df_long_filtered.write.format("sqlserver") \
    .option("host", "sqls-nkipermeteo-dev.database.windows.net") \
    .option("database", "db-nkipermeteo") \
    .option("dbtable", "[lf-ogd-smn_d_recent]") \
    .option("user", dbutils.secrets.get(scope="meteoschweiz", key="sql-username")) \
    .option("password", dbutils.secrets.get(scope="meteoschweiz", key="sql-password")) \
    .option("batchsize", 10000) \
    .option("numPartitions", 1) \
    .mode("append") \
    .save()